# DeepVAR — Deep Vector Autoregression (Turkey, Pipeline B)

**Model**: PyTorch multivariate LSTM (DeepVAR equivalent)  
**Target**: `ngdprsaxdctrq`  
**Variables**: Cat3 (22 vars + 3 COVID = 25 total)  
**Train**: 1995-Q2 to 2011-Q4 / **Val**: 2012-Q1 to 2017-Q4 / **Test**: 2018-Q1 to 2025-Q4 (32 quarters)  
**Scaling**: YES (StandardScaler per variable before sequence construction)  
**HP tuning**: NO (architecture fixed to match US pipeline: num_layers=4, num_cells=40, context_length=12)  
**Note**: Parallel to US model_deepvar.ipynb. Same PyTorch LSTM architecture — data-agnostic, no structural deviation needed.  
Architecture justification: Salinas et al. (2020, DeepAR/DeepVAR), same hyperparams as US benchmark for cross-country comparability.
**Runtime**: ~10-30s per vintage on Turkey's shorter history. Full test loop ~20-60 min.


In [1]:
import sys, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, os.path.join(os.path.pardir, 'turkey_data'))
from turkey_helpers import (
    gen_lagged_data, gen_vintage_data, mean_fill_dataset, get_features,
    load_data, PREDICTIONS_DIR, TARGET, COVID
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

TRAIN_START    = '1995-01-01'
TRAIN_END      = '2011-12-31'
VAL_END        = '2017-12-31'
TEST_START     = '2018-01-01'
TEST_END       = '2025-12-31'
CONTEXT_LENGTH = 12   # matches US pipeline (Salinas et al. 2020 context window)
NUM_LAYERS     = 4    # matches US pipeline
NUM_CELLS      = 40   # matches US pipeline
DROPOUT        = 0.01 # matches US pipeline
EPOCHS         = 10   # matches US pipeline
LR             = 1e-4 # matches US pipeline
BATCH_SIZE     = 100  # matches US pipeline
VINTAGES       = {'m1': -2, 'm2': -1, 'm3': 0, 'post1': 1, 'post2': 2}

DEVICE = torch.device('cpu')

print('DeepVAR (PyTorch, Turkey) — Cat3=25, context_length={}'.format(CONTEXT_LENGTH))


class DeepVARNet(nn.Module):
    """Multivariate LSTM: takes [B, T, n_vars] -> predicts [B, n_vars] one step ahead."""
    def __init__(self, n_vars, hidden=40, n_layers=4, dropout=0.01):
        super().__init__()
        self.lstm = nn.LSTM(n_vars, hidden, n_layers, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden, n_vars)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


def make_sequences(arr, context_length):
    """Slide context window over T x n matrix; X=[B,T,n], y=[B,n] (next step)."""
    X, y = [], []
    for t in range(context_length, len(arr)):
        X.append(arr[t - context_length:t])
        y.append(arr[t])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def train_deepvar(arr_scaled, context_length, n_vars, epochs, lr, batch_size, seed):
    torch.manual_seed(seed)
    X, y = make_sequences(arr_scaled, context_length)
    if len(X) == 0:
        return None
    ds = torch.utils.data.TensorDataset(torch.tensor(X), torch.tensor(y))
    dl = torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=True)
    model = DeepVARNet(n_vars, NUM_CELLS, NUM_LAYERS, DROPOUT).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    model.train()
    for _ in range(epochs):
        for xb, yb in dl:
            opt.zero_grad()
            loss_fn(model(xb.to(DEVICE)), yb.to(DEVICE)).backward()
            opt.step()
    return model


def predict_deepvar(model, arr_scaled, context_length):
    model.eval()
    ctx = torch.tensor(arr_scaled[-context_length:][np.newaxis], dtype=torch.float32)
    with torch.no_grad():
        pred_scaled = model(ctx.to(DEVICE)).cpu().numpy()[0]
    return pred_scaled


print('Model class defined.')


DeepVAR (PyTorch, Turkey) — Cat3=25, context_length=12
Model class defined.


In [2]:
monthly, _, metadata = load_data()
features = get_features('cat3', with_covid=True)
print('Features: {} (Cat3 + COVID)'.format(len(features)))

# DeepVAR is multivariate: predict TARGET + all Cat3 vars 1-step-ahead.
# TARGET (ngdprsaxdctrq) is NOT in get_features(), add explicitly.
all_vars   = [TARGET] + features
target_idx = 0
var_cols   = ['date'] + all_vars

test_quarters = monthly[
    (monthly['date'] >= TEST_START) &
    (monthly['date'] <= TEST_END) &
    monthly['date'].dt.month.isin([3, 6, 9, 12])
]['date'].tolist()

predictions = {v: [] for v in VINTAGES}
actuals_list = []
failed = 0

for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actual = monthly[monthly['date'] == pd_q][TARGET].values[0]
    actuals_list.append(actual)

    base = monthly[(monthly['date'] >= TRAIN_START) & (monthly['date'] <= pd_q)].reset_index(drop=True)
    base = base[var_cols]

    train_m = gen_vintage_data(metadata, base.copy(), pd_q, pd_q)
    train_m[TARGET] = train_m[TARGET].shift(1)
    train_filled = mean_fill_dataset(train_m, train_m)

    arr_train = train_filled[all_vars].values.astype(np.float32)
    scaler = StandardScaler()
    arr_scaled = scaler.fit_transform(arr_train)

    try:
        model = train_deepvar(arr_scaled, CONTEXT_LENGTH, len(all_vars),
                              EPOCHS, LR, BATCH_SIZE, SEED)
        if model is None:
            raise ValueError('Not enough data')
    except Exception:
        for vn in VINTAGES:
            predictions[vn].append(np.nan)
        failed += len(VINTAGES)
        continue

    for vn, offset in VINTAGES.items():
        vintage_date = pd_q + pd.DateOffset(months=offset)
        test_m = gen_vintage_data(metadata, base.copy(), pd_q, vintage_date)
        test_m[TARGET] = test_m[TARGET].shift(1)
        test_filled = mean_fill_dataset(train_m, test_m)

        try:
            arr_test = test_filled[all_vars].values.astype(np.float32)
            arr_test_scaled = scaler.transform(arr_test)
            pred_scaled = predict_deepvar(model, arr_test_scaled, CONTEXT_LENGTH)
            pred_all = scaler.inverse_transform(pred_scaled[np.newaxis])[0]
            pred = pred_all[target_idx]
        except Exception:
            pred = np.nan
            failed += 1

        predictions[vn].append(pred)

    if (i + 1) % 4 == 0:
        print('  {} / {}'.format(i + 1, len(test_quarters)))

print('Done. {} quarters, {} failures.'.format(len(test_quarters), failed))


Features: 25 (Cat3 + COVID)


  4 / 32


  8 / 32


  12 / 32


  16 / 32


  20 / 32


  24 / 32


  28 / 32


  32 / 32
Done. 32 quarters, 0 failures.


In [3]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
for vn in VINTAGES:
    pd.DataFrame({
        'date': test_quarters, 'actual': actuals_list,
        'prediction': predictions[vn],
    }).to_csv(os.path.join(PREDICTIONS_DIR, 'deepvar_tr_{}.csv'.format(vn)), index=False)
    print('Saved deepvar_tr_{}.csv'.format(vn))


Saved deepvar_tr_m1.csv
Saved deepvar_tr_m2.csv
Saved deepvar_tr_m3.csv
Saved deepvar_tr_post1.csv
Saved deepvar_tr_post2.csv


In [4]:
def rmse(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.sqrt(np.mean((a[m] - p[m]) ** 2)) if m.sum() > 0 else np.nan

def mae(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.mean(np.abs(a[m] - p[m])) if m.sum() > 0 else np.nan

panels = {
    'Pre-crisis (2018-2019)': ('2018-01-01', '2019-12-31'),
    'COVID      (2020-2021)': ('2020-04-01', '2021-12-31'),
    'Post-COVID (2022-2025)': ('2022-01-01', '2025-12-31'),
    'Full test  (2018-2025)': ('2018-01-01', '2025-12-31'),
}
a = np.array(actuals_list)
d = pd.to_datetime(test_quarters)
print('DeepVAR (Turkey) — Evaluation by Panel & Vintage')
for pn, (ps, pe) in panels.items():
    m = (d >= ps) & (d <= pe)
    print('--- {} ---'.format(pn))
    for vn in VINTAGES:
        pp = np.array(predictions[vn])
        print('  {}  RMSFE={:.6f}  MAE={:.6f}  N={}'.format(
            vn, rmse(a[m], pp[m]), mae(a[m], pp[m]), m.sum()))


DeepVAR (Turkey) — Evaluation by Panel & Vintage
--- Pre-crisis (2018-2019) ---
  m1  RMSFE=0.018414  MAE=0.013606  N=8
  m2  RMSFE=0.018408  MAE=0.013602  N=8
  m3  RMSFE=0.018408  MAE=0.013604  N=8
  post1  RMSFE=0.018409  MAE=0.013604  N=8
  post2  RMSFE=0.018408  MAE=0.013603  N=8
--- COVID      (2020-2021) ---
  m1  RMSFE=0.069320  MAE=0.042949  N=7
  m2  RMSFE=0.069305  MAE=0.042938  N=7
  m3  RMSFE=0.069306  MAE=0.042937  N=7
  post1  RMSFE=0.069306  MAE=0.042937  N=7
  post2  RMSFE=0.069307  MAE=0.042937  N=7
--- Post-COVID (2022-2025) ---
  m1  RMSFE=0.011941  MAE=0.008366  N=16
  m2  RMSFE=0.011939  MAE=0.008362  N=16
  m3  RMSFE=0.011939  MAE=0.008360  N=16
  post1  RMSFE=0.011939  MAE=0.008359  N=16
  post2  RMSFE=0.011939  MAE=0.008358  N=16
--- Full test  (2018-2025) ---
  m1  RMSFE=0.034779  MAE=0.017251  N=32
  m2  RMSFE=0.034771  MAE=0.017245  N=32
  m3  RMSFE=0.034771  MAE=0.017245  N=32
  post1  RMSFE=0.034772  MAE=0.017244  N=32
  post2  RMSFE=0.034772  MAE=0.017243